# Detecção de Objetos

Nesta aula, avançaremos para a **Detecção de Objetos**, onde podemos ter de zero a *vários* objetos na imagem simultaneamente. Para resolver este problema, implementaremos uma arquitetura inspirada no **YOLOv1 (You Only Look Once)**, que processa a imagem de uma só vez usando uma rede totalmente convolucional e prevê diretamente as *bounding boxes* e as classes dos objetos.

## A Grade (Grid) do YOLO
A ideia do YOLO é dividir a imagem em uma grade (grid) espacial $S \times S$. Cada célula da grade torna-se responsável por detectar o objeto cujo **centro** cai exatamente dentro de seus limites.

Para simplificar nosso modelo didático, definiremos uma grade $7 \times 7$ ($S=7$) e faremos com que cada célula seja capaz de prever **apenas 1** *bounding box* ($B=1$) e as probabilidades de **10** classes ($C=10$, do MNIST). Dessa forma, a saída da nossa rede será um tensor de dimensões $7 \times 7 \times (C + 5 \times B) = 7 \times 7 \times 15$.

As 5 previsões relacionadas à Bounding Box em cada célula são:
- $p_c$: A probabilidade/confiança de conter um objeto.
- $x, y$: As coordenadas do centro do objeto, relativas à célula atual (valores entre 0 e 1).
- $w, h$: A largura e a altura do objeto, relativas ao tamanho total da imagem (valores entre 0 e 1).

In [ ]:
import os
import math
import glob
import random
from PIL import Image
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from tqdm.notebook import tqdm

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## Dataset YOLO e Construção dos Alvos

Na localização simples, o nosso Dataset retornava apenas uma bounding box (um vetor). Na detecção, as imagens podem ter quantidades variáveis de objetos. Portanto, precisamos transformar essa lista variável de anotações (do `.txt`) em um tensor alvo de tamanho fixo: a **grade de predição (Target Grid)**.

Para cada objeto na imagem:
1. Encontramos a qual célula $(i, j)$ seu centro pertence calculando $i = \lfloor y \times S \rfloor$ e $j = \lfloor x \times S \rfloor$.
2. Ativamos a confiança do objeto definindo $p_c = 1$.
3. Adicionamos o *one-hot encoding* da sua classe.
4. Convertamos as coordenadas globais $x_{img}, y_{img}$ para locais da célula: 
   $$ x_{cel} = x_{img} \times S - j $$
   $$ y_{cel} = y_{img} \times S - i $$
5. Mantemos largura e altura em relação à imagem inteira.

In [ ]:
class YOLODataset(Dataset):
    def __init__(self, data_dir, S=7, B=1, C=10, transform=None):
        self.images_dir = os.path.join(data_dir, 'images')
        self.labels_dir = os.path.join(data_dir, 'labels')
        self.transform = transform
        self.S = S
        self.B = B
        self.C = C
        self.img_files = sorted(os.listdir(self.images_dir))
        
    def __len__(self):
        return len(self.img_files)
    
    def __getitem__(self, idx):
        img_name = self.img_files[idx]
        img_path = os.path.join(self.images_dir, img_name)
        label_path = os.path.join(self.labels_dir, img_name.replace('.png', '.txt'))
        
        image = Image.open(img_path).convert('RGB')
        boxes = []
        
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f:
                    class_label, x, y, width, height = [float(val) for val in line.strip().split()]
                    boxes.append([int(class_label), x, y, width, height])
        
        if self.transform:
            image = self.transform(image)
            
        # Matriz Alvo [S, S, C + 5*B]
        label_matrix = torch.zeros((self.S, self.S, self.C + 5 * self.B))
        for box in boxes:
            class_label, x, y, width, height = box
            class_label = int(class_label)
            
            i, j = int(self.S * y), int(self.S * x)
            
            x_cell = self.S * x - j
            y_cell = self.S * y - i
            
            if label_matrix[i, j, self.C] == 0:
                label_matrix[i, j, class_label] = 1 # Classe
                label_matrix[i, j, self.C] = 1      # p_c = 1
                label_matrix[i, j, self.C+1:self.C+5] = torch.tensor([x_cell, y_cell, width, height])
                
        return image, label_matrix

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = YOLODataset('data/mnist_detection/train', transform=transform)
val_dataset = YOLODataset('data/mnist_detection/val', transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

imgs, targets = next(iter(train_loader))
print(f'Shape Imagens: {imgs.shape} | Shape Targets: {targets.shape}')

## Arquitetura

Modelos de detecção eficientes, como as implementações reais do YOLO, descartam as tradicionais camadas densas (`nn.Linear`) na parte final da rede. Em vez disso, utilizam uma **arquitetura totalmente convolucional**, empilhando convoluções para reduzir a resolução espacial enquanto aumentam o número de filtros.

A principal vantagem disso é que as características espaciais (relacionamentos de pixels visuais de onde o dígito está localizado) não são "destruídas" por um *flatten*, sendo preservadas nas saídas finais.

O nosso `YOLONet` terminará em uma camada convolucional $1 \times 1$, que funciona de forma semelhante a uma camada Linear operando independentemente em cada posição $i,j$ da grade espacial, reduzindo os canais para exatamente $C + 5B$ representações.

In [ ]:
class YOLONet(nn.Module):
    def __init__(self, S=7, B=1, C=10):
        super().__init__()
        self.S = S
        self.B = B
        self.C = C
        
        def block(in_channels, out_channels, kernel_size, stride, padding):
            return nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding, bias=False),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(inplace=True)
            )
        
        # Backbone convolucional: Reduz a resolução de 224x224 para 7x7 usando stride e pooling.
        self.features = nn.Sequential(
            block(3, 16, 7, 2, 3),    # Saída: 112x112
            nn.MaxPool2d(2, 2),       # Saída: 56x56
            block(16, 32, 3, 1, 1),
            nn.MaxPool2d(2, 2),       # Saída: 28x28
            block(32, 64, 3, 1, 1),
            nn.MaxPool2d(2, 2),       # Saída: 14x14
            block(64, 128, 3, 1, 1),
            nn.MaxPool2d(2, 2)        # Saída: 7x7
        )
        
        # Cabeça preditiva totalmente convolucional (Sem Fully Connected)
        self.head = nn.Sequential(
            block(128, 256, 3, 1, 1),
            block(256, 256, 3, 1, 1),
            # Camada convolucional 1x1 para gerar os (C + 5B) filtros
            nn.Conv2d(256, C + B * 5, kernel_size=1, stride=1, padding=0)
        )
        
    def forward(self, x):
        x = self.features(x)   # (Batch, 128, 7, 7)
        x = self.head(x)       # (Batch, 15, 7, 7)
        
        # Permutamos os canais para que a dimensão da característica fique por último, 
        # facilitando a lógica da loss e a indexação (Batch, 7, 7, 15)
        x = x.permute(0, 2, 3, 1)
        
        # Aplicamos Sigmoid para restringir todas as previsões ao intervalo [0, 1]. 
        # Em modelos reais, usam-se representações logarítmicas e logits sem ativação,
        # mas Sigmoid facilita a demonstração do Erro Médio Quadrático.
        return torch.sigmoid(x)

## Função de Perda Multicomponente

Ao usar uma grade $7 \times 7$, a maior parte das células não terá nenhum dígito do MNIST nelas. Isso gera um desbalanceamento: a rede preverá rapidamente que todos os $p_c=0$, dominando os gradientes de treino e impedindo a rede de prever as bounding boxes adequadamente.

Para resolver isso, o YOLOv1 divide o cálculo da *Loss* (Erro) utilizando o **Erro Médio Quadrático (MSE)** separado em múltiplas frentes e aplica máscaras (`exists_box`) para penalizar coisas diferentes onde há objetos vs onde não há:

$$ Loss_{YOLO} = \lambda_{coord} \sum_{i=0}^{S^2} 1_{obj}^{i} [ (x_i - \hat{x}_i)^2 + (y_i - \hat{y}_i)^2 + (w_i - \hat{w}_i)^2 + (h_i - \hat{h}_i)^2 ] $$
$$ + \sum_{i=0}^{S^2} 1_{obj}^{i} (p_i - \hat{p}_i)^2 $$
$$ + \lambda_{noobj} \sum_{i=0}^{S^2} 1_{noobj}^{i} (p_i - \hat{p}_i)^2 $$
$$ + \sum_{i=0}^{S^2} 1_{obj}^{i} \sum_{c \in classes} (c_i - \hat{c}_i)^2 $$

Resumindo:
1. **Box Loss**: O erro apenas nas células ondem **tem** objeto. Multiplicado por $\lambda_{coord} = 5$ para forçar o modelo a focar nas coordenadas.
2. **Object Loss**: O erro de confiança de objeto ($p_c$) nas células que **tem** objeto (queremos que $\hat{p}_i$ convirja para 1).
3. **No-Object Loss**: O erro de confiança ($p_c$) na vasta maioria das células que **não tem** objeto. Multiplicado por um $\lambda_{noobj} = 0.5$ para reduzir o poder do desbalanceamento.
4. **Class Loss**: O erro na predição de classe (de 0 a 9) **apenas** nas células onde existe um objeto detectado.

In [ ]:
class YoloLoss(nn.Module):
    def __init__(self, S=7, B=1, C=10):
        super().__init__()
        self.mse = nn.MSELoss(reduction='sum')
        self.S = S
        self.B = B
        self.C = C
        self.lambda_noobj = 0.5
        self.lambda_coord = 5.0
        
    def forward(self, predictions, target):
        # Máscara booleana: extrai os targets de p_c na última dimensão
        exists_box = target[..., self.C].unsqueeze(3)
        
        # ====== 1. Box Coordinates Loss ======
        box_predictions = exists_box * predictions[..., self.C+1:]
        box_targets = exists_box * target[..., self.C+1:]
        box_loss = self.mse(box_predictions, box_targets)
        
        # ====== 2. Object Confidence Loss ======
        obj_pred = exists_box * predictions[..., self.C:self.C+1]
        obj_targ = exists_box * target[..., self.C:self.C+1]
        object_loss = self.mse(obj_pred, obj_targ)
        
        # ====== 3. No-Object Confidence Loss ======
        noobj_pred = (1 - exists_box) * predictions[..., self.C:self.C+1]
        noobj_targ = (1 - exists_box) * target[..., self.C:self.C+1]
        no_object_loss = self.mse(noobj_pred, noobj_targ)
        
        # ====== 4. Class Loss ======
        class_pred = exists_box * predictions[..., :self.C]
        class_targ = exists_box * target[..., :self.C]
        class_loss = self.mse(class_pred, class_targ)
        
        # Total com multiplicadores (lambdas)
        loss = (
            self.lambda_coord * box_loss
            + object_loss
            + self.lambda_noobj * no_object_loss
            + class_loss
        )
        
        batch_size = predictions.shape[0]
        return loss / batch_size, box_loss / batch_size, object_loss / batch_size, no_object_loss / batch_size, class_loss / batch_size

## Treinamento

Vamos instanciar o modelo e um laço tradicional onde salvaremos as 4 curvas parciais descritas acima em um dicionário.

In [ ]:
model = YOLONet().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
criterion = YoloLoss()

In [ ]:
num_epochs = 10
history = {'total': [], 'box': [], 'obj': [], 'noobj': [], 'cls': []}

for epoch in range(num_epochs):
    model.train()
    l_total, l_box, l_obj, l_noobj, l_cls = 0, 0, 0, 0, 0
    
    loop = tqdm(train_loader, leave=False)
    for imgs, targets in loop:
        imgs, targets = imgs.to(device), targets.to(device)
        
        predictions = model(imgs)
        loss, box_l, obj_l, noobj_l, cls_l = criterion(predictions, targets)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        l_total += loss.item()
        l_box += box_l.item()
        l_obj += obj_l.item()
        l_noobj += noobj_l.item()
        l_cls += cls_l.item()
        
        loop.set_description(f'Epoch [{epoch+1}/{num_epochs}]')
        loop.set_postfix(loss=loss.item())
        
    N = len(train_loader)
    history['total'].append(l_total / N)
    history['box'].append(l_box / N)
    history['obj'].append(l_obj / N)
    history['noobj'].append(l_noobj / N)
    history['cls'].append(l_cls / N)
    print(f'Epoch [{epoch+1}/{num_epochs}] - Loss: {l_total / N:.4f}')

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(history['total'], label='Total Loss', linewidth=3, linestyle='--')
plt.plot(history['box'], label='Box Loss')
plt.plot(history['obj'], label='Object Loss')
plt.plot(history['noobj'], label='No-Object Loss')
plt.plot(history['cls'], label='Class Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss Value')
plt.legend()
plt.title('Dissecando as curvas de perda de Treinamento')
plt.show()

## Non-Maximum Suppression (NMS)

Como dividimos a imagem em grade e pedimos que cada célula detecte objetos, pode ser que o braço de um dígito adentre a célula vizinha, levando essa vizinha a também tentar detectar o mesmo dígito. A saída será de várias caixas sobrepostas. Para reter apenas a "melhor", usamos o algoritmo de **Non-Maximum Suppression** (Supressão de não-máximos):

1. Ignoramos qualquer predição cuja confiança $p_c < \text{prob\_threshold}$ (ex: menores que 0.3).
2. Pegamos todas as bounding boxes válidas e as **ordenamos** em ordem decrescente (da maior confiança para a menor).
3. Escolhemos a primeira caixa da lista (a de maior $p_c$), a consideramos correta (adicionamos na lista final de saídas) e retiramos ela da lista temporária.
4. Comparamos todas as outras caixas remanescentes com a caixa que acabamos de escolher via índice de **IoU** (Interseção sobre União).
5. Se uma caixa tiver o IoU alto (ex: $> 0.5$) com a nossa escolhida, isso significa que elas se sobrepõem muito e muito provavelmente se referem ao mesmo dígito! Portanto, nós **deletamos** essa caixa vizinha pois a nossa (de maior score) é superior.
6. Repetimos o loop buscando as próximas que restarem.

In [ ]:
def calculate_iou(box1, box2):
    # Recebe x, y, w, h e retorna a interseção
    x1_min, y1_min = box1[0] - box1[2]/2, box1[1] - box1[3]/2
    x1_max, y1_max = box1[0] + box1[2]/2, box1[1] + box1[3]/2
    x2_min, y2_min = box2[0] - box2[2]/2, box2[1] - box2[3]/2
    x2_max, y2_max = box2[0] + box2[2]/2, box2[1] + box2[3]/2
    
    inter_w = max(0, min(x1_max, x2_max) - max(x1_min, x2_min))
    inter_h = max(0, min(y1_max, y2_max) - max(y1_min, y2_min))
    inter_area = inter_w * inter_h
    
    area1 = box1[2] * box1[3]
    area2 = box2[2] * box2[3]
    union_area = area1 + area2 - inter_area + 1e-6
    
    return inter_area / union_area

def non_max_suppression(bboxes, iou_threshold, prob_threshold):
    # bboxes é uma lista: [class_pred, prob_score, x, y, w, h]
    # Passo 1: Filtrar confianças fracas
    bboxes = [box for box in bboxes if box[1] > prob_threshold]
    
    # Passo 2: Ordenar em ordem decrescente pelo score (box[1])
    bboxes = sorted(bboxes, key=lambda x: x[1], reverse=True)
    bboxes_after_nms = []
    
    while bboxes:
        # Passo 3: Escolher a que tem a maior confiança atual e salvá-la
        chosen_box = bboxes.pop(0)
        bboxes_after_nms.append(chosen_box)
        
        # Passo 4 e 5: Retirar qualquer vizinha que sobreponha muito (ou seja, de outra classe) 
        # Aqui mantemos apenas caixas que NÃO sobrepõem muito a chosen_box.
        bboxes = [box for box in bboxes if box[0] != chosen_box[0] or calculate_iou(chosen_box[2:], box[2:]) < iou_threshold]
        
    return bboxes_after_nms

## Inferência e Resultados Finais

A função utilitária a seguir cuida de destrinchar o tensor final previsto $(7, 7, 15)$ em caixas compreensíveis, e os plotamos visualizando sua real efetividade nas imagens de teste.
Em Verde mostramos as marcações corretas, e em Vermelho o que a rede detectou.

In [ ]:
def convert_cellboxes(predictions, S=7):
    batch_size = predictions.shape[0]
    bboxes = []
    
    for b in range(batch_size):
        batch_bboxes = []
        for i in range(S):
            for j in range(S):
                cell_pred = predictions[b, i, j]
                best_class = torch.argmax(cell_pred[:10]).item()
                prob_score = cell_pred[10].item()
                
                x_cell, y_cell = cell_pred[11].item(), cell_pred[12].item()
                w_img, h_img = cell_pred[13].item(), cell_pred[14].item()
                
                x_img = (j + x_cell) / S
                y_img = (i + y_cell) / S
                
                batch_bboxes.append([best_class, prob_score, x_img, y_img, w_img, h_img])
        bboxes.append(batch_bboxes)
    return bboxes

In [ ]:
model.eval()
imgs, targets = next(iter(val_loader))
with torch.no_grad():
    predictions = model(imgs.to(device))

batch_bboxes = convert_cellboxes(predictions.cpu())
batch_targets = convert_cellboxes(targets)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for idx, ax in enumerate(axes.flatten()):
    img_plot = imgs[idx].permute(1, 2, 0) * 0.5 + 0.5
    ax.imshow(img_plot)
    
    true_boxes = non_max_suppression(batch_targets[idx], iou_threshold=0.5, prob_threshold=0.5)
    for box in true_boxes:
        c, p, x, y, w, h = box
        x0, y0 = (x - w/2) * 224, (y - h/2) * 224
        rect = patches.Rectangle((x0, y0), w*224, h*224, linewidth=2, edgecolor='g', facecolor='none')
        ax.add_patch(rect)
    
    pred_boxes = non_max_suppression(batch_bboxes[idx], iou_threshold=0.5, prob_threshold=0.3)
    for box in pred_boxes:
        c, p, x, y, w, h = box
        x0, y0 = (x - w/2) * 224, (y - h/2) * 224
        rect = patches.Rectangle((x0, y0), w*224, h*224, linewidth=2, edgecolor='r', linestyle='--', facecolor='none')
        ax.add_patch(rect)
        ax.text(x0, y0, f'{c} ({p:.2f})', color='white', backgroundcolor='red', fontsize=8)
    
    ax.axis('off')
plt.tight_layout()
plt.show()